# Solar Generation Condition Classification
LearnDepth Academy LLP — Final Capstone

**Important:** This project uses a synthetic educational dataset because an assigned dataset was not available. It must be described as synthetic in the submission.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv('../data/solar_generation_synthetic.csv')
df.head()

In [ ]:
print('Shape:', df.shape)
print('\nMissing values:\n', df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nClass distribution:\n', df['generation_condition'].value_counts())

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='generation_condition')
plt.title('Generation Condition Distribution')
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(data=df, x='irradiance_W_m2', y='solar_output_kW', hue='generation_condition')
plt.title('Irradiance vs Solar Output')
plt.tight_layout(); plt.show()

In [ ]:
clean = df.drop_duplicates().copy()
X = clean.drop(columns=['generation_condition'])
y = clean['generation_condition']
print('Clean shape:', clean.shape)

## Model Development
Use the same preprocessing pipeline for training and prediction.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

numeric_features = ['solar_output_kW','irradiance_W_m2','temperature_C','cloud_indicator']
categorical_features = ['time_of_day']
preprocessor = ColumnTransformer([
 ('num', Pipeline([('imputer', SimpleImputer(strategy='median')),('scaler', StandardScaler())]), numeric_features),
 ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
])
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
models = {'Logistic Regression': LogisticRegression(max_iter=1000,random_state=42), 'KNN': KNeighborsClassifier(n_neighbors=7), 'Decision Tree': DecisionTreeClassifier(max_depth=5,random_state=42)}
results=[]
for name, model in models.items():
    pipe=Pipeline([('preprocessor',preprocessor),('model',model)])
    pipe.fit(X_train,y_train); pred=pipe.predict(X_test)
    results.append([name,accuracy_score(y_test,pred),precision_score(y_test,pred,average='weighted'),recall_score(y_test,pred,average='weighted'),f1_score(y_test,pred,average='weighted')])
pd.DataFrame(results,columns=['Model','Accuracy','Precision','Recall','F1-Score'])